In [ ]:
# Toy Monte Carlo of the analytic model; the physics from full_analytic's first cell
import nbformat
exec(nbformat.read("full_analytic.ipynb", as_version=4).cells[0].source)

N_SHOWERS = 10_000_000 # speedy and good enough
# N_SHOWERS = 100_000_000 # for final plots

rng = np.random.default_rng(1)
E_a = 1e17                # eV, lowest generated energy
L = 5000.0                # m, side of the square the showers are thrown over

def generate(n):
    """Energies drawn from dJ/du above E_a, and the weight of each shower."""
    a = np.log(E_a / epsilon)
    u = a - np.log(1 - rng.uniform(size=n)) / (gamma - 1)
    return epsilon * np.exp(u), J(E_a) / n

E, w = generate(N_SHOWERS)

In [ ]:
# Shower energies: the weighted showers against the power law
edges = np.logspace(np.log10(E_a), 20, 61)
counts = np.histogram(E, bins=edges)[0]
du = np.diff(np.log(edges))
ec = np.sqrt(edges[1:] * edges[:-1])

fig, ax = plt.subplots(figsize=(7, 5))
ax.errorbar(ec, counts * w / du, yerr=np.sqrt(counts) * w / du, fmt="o", markersize=3, color="k",
            label="simulated showers")
ax.plot(ec, ec * dJdE(ec), "b-", label="$dJ/du$")
ax.plot(ec, (gamma - 1) * J(ec), "r--", label="$(\\gamma - 1) \\, J$")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("E (eV)"); ax.set_ylabel("E dJ/dE (1/m$^2$ sr s)")
ax.set_title("Simulated shower energies")
ax.legend(); plt.tight_layout(); plt.savefig("pdf/shower_energy_sim.pdf"); plt.show()

In [ ]:
# Inclusive shower rate: showers per km^2 per year above E
edges = np.logspace(np.log10(E_a), 20, 61)
above = len(E) - np.searchsorted(np.sort(E), edges)
per_km2_yr = w * C * np.pi * 1e6 * YEAR

fig, ax = plt.subplots(figsize=(7, 5))
ax.errorbar(edges, above * per_km2_yr, yerr=np.sqrt(above) * per_km2_yr, fmt="o", markersize=3, color="k",
            label="simulated showers")
ax.plot(edges, C * np.pi * J(edges) * 1e6 * YEAR, "r-", label="model")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("E (eV)"); ax.set_ylabel("showers per km$^2$ per year above E")
ax.set_title("Inclusive shower rate")
ax.legend(); plt.tight_layout(); plt.savefig("pdf/inclusive_shower_rate.pdf"); plt.show()

In [ ]:
# Throw showers over the square; muon density at its centre
x = rng.uniform(-L / 2, L / 2, N_SHOWERS)
y = rng.uniform(-L / 2, L / 2, N_SHOWERS)
r = np.sqrt(x**2 + y**2)
dens = rho(r, E, lmbda)

In [ ]:
# Detectable rate: showers per year above each density, against the closed form
per_yr = w * C * np.pi * L**2 * YEAR
edges = np.linspace(15, 100, 35)                # below about 14, showers under E_a would be missing
counts = np.histogram(dens, bins=edges)[0]
counts[-1] += np.sum(dens > edges[-1])          # overflow into the last bin
above = np.cumsum(counts[::-1])[::-1]           # showers above each lower edge
phi = edges[:-1]

mc, err = above * per_yr, np.sqrt(above) * per_yr
plt.plot(phi, mc, "C0-", label="toy Monte Carlo")
plt.fill_between(phi, mc - err, mc + err, color="C0", alpha=0.3, linewidth=0, label="statistical, 1$\\sigma$")
plt.plot(phi, R(phi) * YEAR, "r-", label="closed form")
for phid in phi_D:
    plt.annotate(f"R = {R(phid) * YEAR:.1f} / yr", xy=(phid, R(phid) * YEAR), xytext=(40, 25),
                 textcoords="offset points", arrowprops=dict(arrowstyle="->"))
plt.yscale("log"); plt.xlabel("muon density (1/m$^2$)"); plt.ylabel("showers per year above")
plt.title("Detectable rate"); plt.legend(); plt.savefig("pdf/toy_detectable_rate.pdf"); plt.show()

for phid in phi_D:
    n = np.sum(dens > phid)
    print(f"Phi_D {phid:g}:  toy MC {n * per_yr:.2f} +- {np.sqrt(n) * per_yr:.2f} / yr,  closed form {R(phid) * YEAR:.2f}")